In [1]:
# 02_risk_metrics.ipynb
# 목적: VaR, CVaR, 샤프지수, MDD 계산 모듈

import pandas as pd
import numpy as np
import os

# 저장된 데이터 불러오기
data_path = os.path.expanduser("~/Desktop/SentriVaR-500/data")
df = pd.read_csv(f"{data_path}/prices.csv", index_col="Date", parse_dates=True)

# 일별 수익률 계산
returns = df.pct_change().dropna()

print(f"데이터 로드 완료: {df.shape}")
print(f"기간: {df.index[0].date()} ~ {df.index[-1].date()}")

데이터 로드 완료: (2138, 5)
기간: 2018-01-02 ~ 2026-07-07


In [2]:
# 리스크 지표 계산 함수로 정리 (재사용 가능하게)
def calculate_risk_metrics(return_series, confidence=0.95):
    """
    단일 종목의 리스크 지표 계산
    - return_series: 일별 수익률 Series
    - confidence: VaR 신뢰구간 (기본 95%)
    """
    # VaR: 95% 확률로 하루 최대 손실
    VaR = return_series.quantile(1 - confidence)

    # CVaR: VaR 초과 손실의 평균
    CVaR = return_series[return_series <= VaR].mean()

    # 샤프지수: 리스크 대비 수익률 (연간화)
    sharpe = return_series.mean() / return_series.std() * np.sqrt(252)

    # MDD: 고점 대비 최대 낙폭
    cumulative = (1 + return_series).cumprod()
    rolling_max = cumulative.cummax()
    drawdown = (cumulative - rolling_max) / rolling_max
    mdd = drawdown.min()

    return {
        "VaR (95%)": round(VaR, 4),
        "CVaR (95%)": round(CVaR, 4),
        "Sharpe Ratio": round(sharpe, 4),
        "MDD": round(mdd, 4)
    }

# 전체 종목에 적용
results = {}
for ticker in returns.columns:
    results[ticker] = calculate_risk_metrics(returns[ticker])

risk_df = pd.DataFrame(results).T
print(risk_df)

       VaR (95%)  CVaR (95%)  Sharpe Ratio     MDD
AAPL     -0.0297     -0.0438        0.9414 -0.3852
GOOGL    -0.0296     -0.0443        0.8901 -0.4432
JPM      -0.0264     -0.0410        0.7085 -0.4363
MSFT     -0.0283     -0.0407        0.7998 -0.3715
SOXX     -0.0367     -0.0520        0.9388 -0.4575


In [3]:
# 리스크 지표 결과 저장
risk_df.to_csv(f"{data_path}/risk_metrics.csv")
print("저장 완료: risk_metrics.csv")

# 결과 해석
print("\n--- 종목별 분석 ---")
print(f"수익 효율 1위: {risk_df['Sharpe Ratio'].idxmax()} (Sharpe {risk_df['Sharpe Ratio'].max()})")
print(f"리스크 최소: {risk_df['VaR (95%)'].idxmax()} (VaR {risk_df['VaR (95%)'].max()})")
print(f"낙폭 최소: {risk_df['MDD'].idxmax()} (MDD {risk_df['MDD'].max()})")

저장 완료: risk_metrics.csv

--- 종목별 분석 ---
수익 효율 1위: AAPL (Sharpe 0.9414)
리스크 최소: JPM (VaR -0.0264)
낙폭 최소: MSFT (MDD -0.3715)
